# Chapter 5: Decision Trees and Gradient Boosting

## A. Tree-based methods

### 1. Introduction

Tree-based models are a family of ensemble algorithms known for their strong performance on tabular data and minimal preprocessing requirements. These models rely on **decision trees**, which date back to the 1960s. In both classification and regression tasks, decision trees work by splitting the dataset into subsets where predictions are easier—either because the classes are more uniform or the target values are more consistent.

Each decision tree starts at a **root node** and makes a **binary or multi-way split** based on feature conditions. These conditions lead to **branches**, which may either end at **leaf nodes**—where predictions are made—or continue to more decision nodes.

In classification problems, decision trees use **splitting criteria** to choose the best feature and value for creating homogeneous subsets. Common criteria include:


| Criterion            | Formula                                                                         | Purity Focus          | Speed              | Used                                                 | Interpretation                                                  | Extra Notes                                                |
| -------------------- | ------------------------------------------------------------------------------- | --------------------- | ------------------ | ------------------------------------------------------- | --------------------------------------------------------------- | ---------------------------------------------------------- |
| **Gini Impurity**    | $Gini = 1 - \sum p_i^2$                                                         | Medium (quick & good) | Faster             | Balanced classes                          | Measures impurity: how often you'd misclassify a random element | More sensitive to node purity                              |
| **Entropy**          | $Entropy = -\sum p_i \log_2(p_i)$                                               | Slightly higher       | Slower             | Unbalanced classes | Measures "surprise" or uncertainty in class labels              | Information Gain is derived from Entropy                   |



When used for **regression**, decision trees split the data to minimize metrics like:
- **Mean Squared Error (MSE)**
- **Mean Absolute Error (MAE)**
- **Variance** within each subset

The tree automatically selects the best features by evaluating different splits. Once trained, predictions are fast and involve simply traversing the tree from root to leaf.

**Advantages:**
- No need for feature scaling or transformation
- Can model **nonlinear relationships** naturally by partitioning feature space
- Easy to visualize and interpret

**Limitations:**
- Highly **prone to overfitting**
- Can create overly complex trees with many small partitions

**Overfitting Control Techniques:**
- **Limit** the maximum number of splits or depth
- **Prune** the tree after construction to remove weak splits

Fully grown trees can perfectly fit training data but perform poorly on new examples due to overfitting. Simpler trees may underfit but generalize better.

**Key Insight:**  
Decision trees are **high-variance models**. They perform best not as standalone models but as **base learners in ensemble methods** like:
- **Bagging**
- **Random Forests**
- **Gradient Boosting**

We'll demonstrate this next using the Airbnb NYC dataset.


### 2. Import data and packages

In [8]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, accuracy_score
from sklearn.model_selection import KFold, cross_validate



# Import data
data = pd.read_csv("./data/AirBnB.csv")

### 3. Create labels

In [3]:
#A. list of features to be excluded from data processing
excluding_list = ['price', 'id', 'latitude', 'longitude', 'host_id',
                  'last_review', 'name', 'host_name'] 

#B. list of low-cardinality categorical features to be one-hot encoded
low_card_categorical = ['neighbourhood_group', 'room_type'] 

#C. list of high-cardinality categorical features to be ordinally encoded
high_card_categorical = ['neighbourhood']
continuous = ['minimum_nights', 'number_of_reviews', 'reviews_per_month',
              'calculated_host_listings_count', 'availability_365']

#D. creating a binary target indicating whether the price is above the mean (unbalanced binary target)
target_mean = (data["price"] > data["price"].mean()).astype(int) 

#E. creating a binary target indicating whether the price is above the median (balanced binary target)
target_median = (data["price"] > data["price"].median()).astype(int)

#F. creating a multiclass target by quantile binning the price into 5 classes
target_multiclass = pd.qcut(data["price"], q=5, labels=False)

#G. setting the target for regression as the price column
target_regression = data["price"]



### 4. Train a decision tree classifier

In [4]:
#A. creating a column transformer that applies different transformations to categorical and numeric features
categorical_onehot_encoding = OneHotEncoder(handle_unknown='ignore')
categorical_ord_encoding = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan)
numeric_passthrough = SimpleImputer(strategy="constant", fill_value=0)

column_transform = ColumnTransformer(
    [('categories', categorical_onehot_encoding, low_card_categorical),
     ('numeric', numeric_passthrough, continuous)], #A
    remainder='drop',
    verbose_feature_names_out=False,
    sparse_threshold=0.0)

#B. an instance of a decision tree classifier
model = DecisionTreeClassifier(random_state=0) #B


#C a pipeline that sequentially applies column transformation and the decision tree model
model_pipeline = Pipeline(
    [('processing', column_transform),
     ('modeling', model)]) #C


#D a five-fold cross-validation us  ing the defined pipeline, calculating accuracy scores, and returning additional information
accuracy = make_scorer(accuracy_score)
cv = KFold(5, shuffle=True, random_state=0)
cv_scores = cross_validate(estimator=model_pipeline,
                           X=data,
                           y=target_median,
                           scoring=accuracy,
                           cv=cv,
                           return_train_score=True,
                           return_estimator=True) #D

#E. printing the mean and standard deviation of the test accuracy scores from cross-validation
mean_cv = np.mean(cv_scores['test_score'])
std_cv = np.std(cv_scores['test_score'])
fit_time = np.mean(cv_scores['fit_time'])
score_time = np.mean(cv_scores['score_time'])

#E printing the mean and standard deviation of the accuracy scores from cross-validation
print(f"""mean_csv = {mean_cv:0.3f} ({std_cv:0.3f})
fit_time: {fit_time:0.2f} secs 
pred_time: {score_time:0.2f} secs""")


mean_csv = 0.761 (0.005)
fit_time: 0.13 secs 
pred_time: 0.01 secs


### 5. Sampling techniques and esemble models

| **Strategy**         | **Sampling Type**                               | **Goal**               | **Pros**                                      | **Cons**                                   | **Sklearn Params**                                       |
| -------------------- | ----------------------------------------------- | ---------------------- | --------------------------------------------- | ------------------------------------------ | -------------------------------------------------------- |
| **Pasting**          | Data only, **without replacement**              | Reduce variance        | Low memory use, more robust to outliers       | Can increase bias, may miss key data       | `bootstrap=False`, `max_samples<1.0`                     |
| **Bagging**          | Data only, **with replacement** (bootstrapping) | Reduce variance        | Diverse datasets, reduces overfitting         | Still computationally expensive            | `bootstrap=True`, `max_samples=1.0`                      |
| **Random Subspaces** | **Features only**, without replacement          | Reduce variance        | Reduces feature correlation in models         | Bias remains if important features omitted | `bootstrap=False`, `max_features<1.0`                    |
| **Random Patches**   | Both **data and features**, without replacement | Max variance reduction | Maximum model diversity → best generalization | Most complex to tune                       | `bootstrap=False`, `max_samples<1.0`, `max_features<1.0` |
| **Boosting**         | Sequential learning, each learns from previous  | Reduce **bias**        | High accuracy, powerful for complex patterns  | Sensitive to noise, can overfit            | Use `GradientBoosting`, `XGBoost`, etc.                  |

**Note**:
* Pasting, bagging, subspaces, and patches use avergaing for regression and majority vote for classification.
* Boosting uses errors from previous step



In [5]:
#A creating a BaggingClassifier ensemble model based on decision trees
model = BaggingClassifier(estimator=DecisionTreeClassifier(),
                          n_estimators=300,
                          bootstrap=True, #B. setting bootstrap sampling for the BaggingClassifier
                          max_samples=1.0, #C. setting no sampling of features for the BaggingClassifier
                          max_features=1.0, #D. setting no sampling of data for the BaggingClassifier
                          random_state=0)

#E. a column transformer that applies different transformations to categorical and numeric features
column_transform = ColumnTransformer(
    [('categories', categorical_onehot_encoding, low_card_categorical),
     ('numeric', numeric_passthrough, continuous)],
    remainder='drop',
    verbose_feature_names_out=False,
    sparse_threshold=0.0)

#F a pipeline that sequentially applies column transformation and the bagging classifier model
model_pipeline = Pipeline(
    [('processing', column_transform),
     ('modeling', model)]) 

#G a five-fold cross-validation using the defined pipeline and calculating accuracy scores
accuracy = make_scorer(accuracy_score)
cv = KFold(5, shuffle=True, random_state=0)
cv_scores = cross_validate(estimator=model_pipeline,
                           X=data,
                           y=target_median,
                           scoring=accuracy,
                           cv=cv,
                           return_train_score=True,
                           return_estimator=True)

#H printing the mean and standard deviation of the accuracy scores from cross-validation
mean_cv = np.mean(cv_scores['test_score'])
std_cv = np.std(cv_scores['test_score'])
fit_time = np.mean(cv_scores['fit_time'])
score_time = np.mean(cv_scores['score_time'])

print(f"""mean_csv = {mean_cv:0.3f} ({std_cv:0.3f})
fit_time: {fit_time:0.2f} secs 
pred_time: {score_time:0.2f} secs""")

mean_csv = 0.809 (0.004)
fit_time: 21.95 secs 
pred_time: 0.58 secs


### 6. Random Forests


| **Aspect**                  | **Summary**                                                                                                                                                                                              |
| --------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **What it is**              | An ensemble method that combines **bagging** and **random feature selection** (random patches).                                                                                                          |
| **Base model**              | **Decision Tree** (built via binary splits).                                                                                                                                                             |
| **Sampling method**         | - **Bootstrapping** of data (with replacement)<br>- **Subsampling of features** at each split                                                                                                            |
| **Why it works**            | Promotes **diversity** among trees → reduces **variance** → improves generalization                                                                                                                      |
| **Prediction type**         | - **Classification**: Majority voting<br>- **Regression**: Averaging                                                                                                                                     |
| **Advantages**              | - Handles non-linearities well<br>- Resistant to overfitting<br>- Estimates **feature importance**                                                                                                       |
| **Hyperparameters to tune** | - `n_estimators`: Number of trees (more trees = better averaging)<br>- `max_features`: Limits overfitting<br>- `max_depth`: Controls tree size<br>- `min_samples_leaf`: Increases bias, reduces variance |
| **Bias-Variance Tradeoff**  | Tune depth and leaf size to **increase bias** and **control variance**                                                                                                                                   |
| **Scikit-learn class**      | `RandomForestClassifier` or `RandomForestRegressor`                                                                                                                                                      |
| **Extra features**          | - Good **feature importance metrics**<br>- Can measure **case similarity**                                                                                                                               |
| **Best use cases**          | - **Tabular data**, noisy or with many features<br>- When you want fast, robust baseline models                                                                                                          |



> “Random Forests build many uncorrelated decision trees by combining bootstrapping with random feature selection. This ensemble approach reduces variance and overfitting, making it one of the most reliable algorithms for tabular data.”

In [7]:
#A. a RandomForestClassifier with 300 estimators and a minimum number of samples at a leaf node set to 3
model = RandomForestClassifier(n_estimators=300,
                               min_samples_leaf=3,
                               random_state=0) 

#B. a column transformer that applies different transformations to categorical and numeric features
column_transform = ColumnTransformer(
    [('categories', categorical_onehot_encoding, low_card_categorical),
     ('numeric', numeric_passthrough, continuous)],
    remainder='drop',
    verbose_feature_names_out=False,
    sparse_threshold=0.0) 

#C. a pipeline that sequentially applies column transformation and the random forest classifier model
model_pipeline = Pipeline(
    [('processing', column_transform),
     ('modeling', model)]) #C

#D. a five-fold cross-validation using the defined pipeline and calculating accuracy scores
accuracy = make_scorer(accuracy_score)
cv = KFold(5, shuffle=True, random_state=0)
cv_scores = cross_validate(estimator=model_pipeline,
                           X=data,
                           y=target_median,
                           scoring=accuracy,
                           cv=cv,
                           return_train_score=True,
                           return_estimator=True)

#E. printing the mean and standard deviation of the accuracy scores from cross-validation
mean_cv = np.mean(cv_scores['test_score'])
std_cv = np.std(cv_scores['test_score'])
fit_time = np.mean(cv_scores['fit_time'])
score_time = np.mean(cv_scores['score_time'])

print(f"""mean_csv = {mean_cv:0.3f} ({std_cv:0.3f})
fit_time: {fit_time:0.2f} secs 
pred_time: {score_time:0.2f} secs""")

mean_csv = 0.826 (0.004)
fit_time: 7.58 secs 
pred_time: 0.46 secs


### 7. EXtremely Randomized Trees (ERTs)




| **Aspect**                 | **Summary**                                                                                 |
| -------------------------- | ------------------------------------------------------------------------------------------- |
| **What it is**             | A more **randomized variant** of Random Forests                                             |
| **Key difference**         | Instead of selecting the best feature for a split(using *Gini* or *Entropy*),<br> it **chooses a feature at random**       |
| **Split logic**            | - Randomly select a feature<br>- Find best split **within that feature only**               |
| **Why it works**           | - Trees are **highly uncorrelated**<br>- Better **variance reduction**<br>- Faster to train |
| **Bias–Variance Tradeoff** | - **Higher bias** (less optimal splits)<br>- **Lower variance** (more randomness)           |
| **Best use cases**         | - **High-dimensional** data<br>- **Noisy** datasets<br>- **Imbalanced** classes             |
| **Performance**            | Slightly better or comparable to Random Forests, but **much faster**                        |
| **Scikit-learn class**     | `ExtraTreesClassifier` or `ExtraTreesRegressor`                                             |
| **Hyperparameters**        | Same as Random Forests:<br>`n_estimators`, `max_features`, `max_depth`, `min_samples_leaf`  |
| **Efficiency**             | Faster because it **skips feature comparisons**                                             |
| **Interpretation**         | Less likely to overfit noise; naturally ignores collinearity                                |
| **Drawback**               | Less optimal splits can slightly hurt accuracy in low-noise, low-dimensional settings       |



In [9]:
#A. an ExtraTreesClassifier with 300 estimators and a minimum number of samples at a leaf node set to 3
model = ExtraTreesClassifier(n_estimators=300,
                             min_samples_leaf=3,
                             random_state=0) #A

#B. a column transformer that applies different transformations to categorical and numeric features
column_transform = ColumnTransformer(
    [('categories', categorical_onehot_encoding, low_card_categorical),
     ('numeric', numeric_passthrough, continuous)],
    remainder='drop',
    verbose_feature_names_out=False,
    sparse_threshold=0.0) #B

#C. a pipeline that sequentially applies column transformation and the random forest classifier model
model_pipeline = Pipeline(
    [('processing', column_transform),
     ('modeling', model)]) #C

#D a five-fold cross-validation using the defined pipeline and calculating accuracy scores
accuracy = make_scorer(accuracy_score)
cv = KFold(5, shuffle=True, random_state=0)
cv_scores = cross_validate(estimator=model_pipeline,
                           X=data,
                           y=target_median,
                           scoring=accuracy,
                           cv=cv,
                           return_train_score=True,
                           return_estimator=True) #D

#E printing the mean and standard deviation of the accuracy scores from cross-validation
mean_cv = np.mean(cv_scores['test_score'])
std_cv = np.std(cv_scores['test_score'])
fit_time = np.mean(cv_scores['fit_time'])
score_time = np.mean(cv_scores['score_time'])

print(f"""mean_csv = {mean_cv:0.3f} ({std_cv:0.3f})
fit_time: {fit_time:0.2f} secs 
pred_time: {score_time:0.2f} secs""")

mean_csv = 0.823 (0.004)
fit_time: 2.88 secs 
pred_time: 0.28 secs


## B. Gradient boosting

| **Aspect**                    | **Details**                                                                                                                                                                                                                              |
| ----------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **What It Is**                | An ensemble method that builds decision trees sequentially, each correcting the previous one’s errors using gradient descent.                                                                                                            |
| **Use Cases**                 | Multiclass classification, click prediction, search ranking — excels in **tabular data** problems.                                                                                                                                       |
| **Performance**               | Outperforms SVMs, neural nets, bagging, and random forests (when tuned properly).                                                                                                                                                        |
| **Why It's Powerful**         | - Handles **heterogeneous features** without preprocessing<br>- Captures **non-linear relationships**<br>- Robust to **outliers** and **missing data**<br>- **Auto-selects** and ranks features<br>- **Custom loss functions** supported |
| **Base Learner**              | Uses **regression trees**, even for classification (e.g., `DecisionTreeRegressor` in sklearn).                                                                                                                                           |
| **Training Logic**            | - Starts with a **constant prediction**<br>- Each new tree fits the **residuals (errors)** of the ensemble<br>- Learns via **gradient descent on the loss function**                                                                     |
| **Compared to AdaBoost**      | - AdaBoost reweights samples to focus on errors<br>- GBDT fits trees on gradients of the loss function (stronger, more flexible)                                                                                                         |
| **Probabilities**             | GBDT gives **well-calibrated probabilities** (better than Random Forests), useful in fraud, credit risk, medical use cases.                                                                                                              |
| **Key Hyperparameters**       | - `learning_rate (ν)`<br>- `n_estimators (M)`<br>- Tree parameters: `max_depth`, `min_samples_split`, `min_samples_leaf`                                                                                                                 |
| **Optimization**              | Two levels:<br>1. Each tree reduces local error<br>2. Full ensemble updates in **gradient-descent style** toward best predictions                                                                                                        |
| **Loss Function Flexibility** | Can optimize **custom loss functions**, unlike Random Forests (which use Gini/Entropy for classification).                                                                                                                               |


---
<div style="text-align: center;">
  <img src="./data/Boosting.png" alt="Boost" width="600"/>
</div>

---




## C. Boosting in Scikit-learn

## D. Using XGBoost

## E. Introduction to LightGBM

## F. Summary

Sure! Here's a simplified and interview-ready comparison table for **XGBoost** and **LightGBM**, showing their key characteristics and differences. This builds on your Gradient Boosting Decision Tree (GBDT) summary.

---

### 🔍 Comparison Table: XGBoost vs LightGBM (LGBM)

| Feature                        | XGBoost (Extreme Gradient Boosting)                       | LightGBM (Light Gradient Boosting Machine)                  |
| ------------------------------ | --------------------------------------------------------- | ----------------------------------------------------------- |
| **Core Idea**                  | Optimized version of GBDT with regularization and pruning | GBDT with faster training using histogram-based methods     |
| **Speed**                      | Slower than LightGBM                                      | Faster due to histogram binning and leaf-wise growth        |
| **Tree Growth**                | Level-wise (depth-wise), grows trees evenly               | Leaf-wise, grows trees toward leaves with highest loss      |
| **Memory Usage**               | Higher due to exact greedy splitting                      | Lower due to histogram optimization                         |
| **Accuracy**                   | High (typically better than Random Forests)               | Very high, often better than XGBoost when tuned             |
| **Handling Large Datasets**    | Good, but slower on very large data                       | Excellent, especially on large-scale data                   |
| **Categorical Features**       | Must be manually encoded (e.g. one-hot)                   | Native support for categorical features                     |
| **Parallelism**                | Yes – supports multi-core training                        | Yes – more efficient parallel computation                   |
| **Regularization**             | L1 & L2 (helps prevent overfitting)                       | L1 & L2 (similar to XGBoost)                                |
| **Missing Value Handling**     | Handled automatically                                     | Handled automatically                                       |
| **GPU Support**                | Yes                                                       | Yes                                                         |
| **Hyperparameter Sensitivity** | Less sensitive than LightGBM                              | More sensitive – needs careful tuning                       |
| **Used in**                    | Kaggle, production systems                                | Kaggle, production systems, large-scale recommender systems |

---

### ⚙️ Tips for Interviews

* **XGBoost**: Great if you want robust performance with interpretability and regularization.
* **LightGBM**: Ideal when speed and scalability matter (e.g., real-time systems, huge datasets).
* **Both** are usually better than Random Forests and classic boosting for structured/tabular data.

Would you like me to add **CatBoost** to this table too? It's another GBDT variant, great for categorical features.


## Ensemble Algorithms

Ensemble algorithms improve the predictive power of a single model by using multiple models or chaining them together.

- Ensemble algorithms are often based on decision trees.
- There are two core ensemble strategies: **averaging** and **boosting**.
- Averaging strategies, such as random forests, tend to reduce the variance of predictions while only slightly increasing the bias.
- **Pasting** is a type of averaging approach that involves creating a set of different models trained on subsamples of the data and pooling the predictions together.
- **Bagging** is similar to averaging but with bootstrapping instead of subsampling.
- Averaging methods can be computationally intensive and may increase bias by excluding important parts of the data distribution through sampling.

---

### Random Forests

Random forests are an ensemble learning algorithm that combines decision trees by **bootstrapping samples** and **subsampling features** during modeling (random patches).

- Creates a set of models that are different from each other and produces more reliable and accurate predictions.
- Can be used to determine feature importance and measure cases’ similarity in a dataset.
- Requires fine-tuning of hyperparameters like:
  - Number of trees
  - Maximum number of features used for splits
  - Maximum tree depth
  - Minimum size of terminal branches
- Can be computationally costly if the number of trees is too high.

---

### Extremely Randomized Trees (ERT)

ERT is a variation of the random forests algorithm.

- Randomly selects the feature for the split at each decision tree node.
- Leads to less variance (because trees are more diverse) but more bias (due to randomization sacrificing some predictive accuracy).
- More computationally efficient and useful for large datasets with many collinear and noisy features.
- Reduces variance by making the resulting set of trees less correlated.

---

### Gradient Boosted Decision Trees (GBDT)

GBDT is a highly effective machine learning method for tabular data problems, widely used in:

- Multiclass classification
- Advertising click prediction
- Search engine ranking

Compared to methods like neural networks, SVMs, random forests, and bagging ensembles, GBDT often performs better in standard tabular problems.

**Why it works:**

- Combines **gradient descent** (an optimization procedure typical in linear models and neural networks) with **decision trees** trained on the gradients from the sum of the previous trees.

**Scikit-learn**:

- Offers gradient boosting algorithms for regression and classification.
- Recently replaced the original algorithm with a faster histogram-based version.

---

### XGBoost

XGBoost is a gradient boosting decision tree algorithm that gained popularity after winning the **Higgs Boson Machine Learning Challenge** on Kaggle.

- Based on a more complex optimization using **Newton’s Descent**.
- Advantages:
  - Handles various input data types
  - Supports customized objective and evaluation functions
  - Automatically handles missing values
  - Supports GPU training
  - Can enforce monotonicity and feature interaction constraints
  - Optimizes multiple cores and cache on standalone machines

---

### LightGBM

LightGBM is a highly efficient gradient boosting decision tree algorithm introduced in 2017 by Guolin Ke and the Microsoft team.

- Designed to be faster and use less memory than traditional GBDT.
- Achieves efficiency through:
  - **Leaf-wise splitting policy**
  - **Exclusive Feature Bundling (EFB)**
- Performance demonstrated on multiple public datasets.
